<a href="https://colab.research.google.com/github/springboardmentor12458j/LiveMeetingSummarize/blob/varshini/Heart_Disease_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q streamlit
!npm install localtunnel


In [2]:
%%writefile app.py
import streamlit as st
import pandas as pd
import base64
import joblib
import numpy as np


# Page config
st.set_page_config(
    page_title="CardioGuard AI - Heart Disease Prediction",
    page_icon="❤️",
    layout="wide"
)


# Function to encode image to base64
@st.cache_data
def get_base64_of_bin_file(bin_file):
    with open(bin_file, 'rb') as f:
        data = f.read()
    return base64.b64encode(data).decode()


# Load background image
img_base64 = get_base64_of_bin_file('heart_bg.jpg')


# Custom CSS for Background and Transparency (Glassmorphism)
st.markdown(f"""
<style>
    /* Background Image */
    .stApp {{
        background-image: url("data:image/jpg;base64,{img_base64}");
        background-size: cover;
        background-position: center;
        background-attachment: fixed;
    }}


    /* Global transparency for all containers */
    [data-testid="stHeader"], [data-testid="stSidebar"], .main {{
        background: transparent !important;
    }}


    /* Sidebar Glassmorphism */
    [data-testid="stSidebar"] {{
        background: rgba(0, 0, 0, 0.4) !important;
        backdrop-filter: blur(10px);
        border-right: 1px solid rgba(255, 255, 255, 0.1);
    }}


    /* Input elements transparency */
    .stTextInput>div>div>input, .stNumberInput>div>div>input, .stSelectbox>div>div>div {{
        background-color: rgba(255, 255, 255, 0.1) !important;
        color: white !important;
        border: 1px solid rgba(255, 255, 255, 0.2) !important;
        backdrop-filter: blur(5px);
    }}


    /* Label colors */
    label, p, h1, h2, h3, h4 {{
        color: white !important;
    }}


    /* Prediction Card Glassmorphism */
    .prediction-card {{
        padding: 25px;
        border-radius: 20px;
        background: rgba(255, 255, 255, 0.05);
        backdrop-filter: blur(15px);
        border: 1px solid rgba(255, 255, 255, 0.1);
        box-shadow: 0 8px 32px 0 rgba(0, 0, 0, 0.37);
        margin-bottom: 20px;
    }}


    /* Buttons styling */
    .stButton>button {{
        width: 100%;
        border-radius: 12px;
        height: 3.5em;
        background: linear-gradient(135deg, #ff4b4b 0%, #ff1e1e 100%);
        color: white;
        font-weight: bold;
        border: none;
        box-shadow: 0 4px 15px rgba(255, 75, 75, 0.3);
        transition: all 0.3s ease;
    }}
    .stButton>button:hover {{
        transform: translateY(-2px);
        box-shadow: 0 6px 20px rgba(255, 75, 75, 0.5);
        background: linear-gradient(135deg, #ff1e1e 0%, #cc0000 100%);
    }}


    /* Multiselect transparency */
    [data-testid="stMultiSelect"] span {{
        background-color: rgba(255, 75, 75, 0.3) !important;
        color: white !important;
    }}

    /* Divider transparency */
    hr {{
        border: 0;
        height: 1px;
        background-image: linear-gradient(to right, rgba(255, 255, 255, 0), rgba(255, 255, 255, 0.75), rgba(255, 255, 255, 0));
    }}


    /* Slider styling */
    .stSlider [data-baseweb="slider"] {{
        background-color: transparent !important;
    }}


</style>
""", unsafe_allow_html=True)


# Load Activity Model
@st.cache_resource
def load_activity_model():
    try:
        model = joblib.load('activity_model.joblib')
        return model
    except:
        return None


activity_model = load_activity_model()


# Mapping activity IDs to names based on MHEALTH dataset
ACTIVITY_MAP = {
    0: "Null/Rest",
    1: "Standing Still",
    2: "Sitting/Relaxing",
    3: "Lying Down",
    4: "Walking",
    5: "Climbing Stairs",
    6: "Waist Bends",
    7: "Frontal Arms Elevation",
    8: "Knees Bending (Crouching)",
    9: "Cycling",
    10: "Jogging",
    11: "Running",
    12: "Jump Front & Back"
}


# Load Risk Model
@st.cache_resource
def load_risk_model():
    try:
        model = joblib.load('risk_model.joblib')
        return model
    except:
        return None


risk_model = load_risk_model()


def get_activity_context(alx, aly, alz):
    if activity_model:
        pred = activity_model.predict([[alx, aly, alz]])[0]
        return ACTIVITY_MAP.get(pred, "Unknown Activity")
    return "Unknown Context"


def predict_heart_disease(bpm, glucose, symptoms, age, activity_context, risk_data):
    results = []
    risks = []

    # Model-based Risk Prediction
    risk_score = 0
    if risk_model:
        # risk_data: [Gender, Chain_smoker, Consumes_other_tobacco_products, HighBP, Obese, Diabetes, Metabolic_syndrome, Use_of_stimulant_drugs, Family_history, History_of_preeclampsia, CABG_history, Respiratory_illness]
        risk_prob = risk_model.predict_proba([risk_data])[0][1] # Probability of "UnderRisk: yes"
        risk_score = round(risk_prob * 100, 1)
    tips = {
        "General": [
            "Maintain a balanced diet rich in fiber, fruits, and vegetables.",
            "Aim for at least 30 minutes of moderate physical activity daily.",
            "Schedule regular check-ups with your cardiologist."
        ],
        "Diabetes (High Glucose)": [
            "Monitor blood sugar levels regularly.",
            "Reduce intake of refined sugars and processed carbohydrates.",
            "Include complex carbs like oats and brown rice in your diet."
        ],
        "Coronary Artery Disease (CAD)": [
            "Adopt a heart-healthy, low-sodium, and low-fat diet.",
            "Quit smoking and avoid secondhand smoke.",
            "Manage stress through meditation or deep-breathing exercises."
        ],
        "Heart Failure": [
            "Restrict daily salt intake to manage fluid retention.",
            "Monitor your daily weight and report sudden changes to your doctor.",
            "Balance rest with light, doctor-approved activity."
        ],
        "Arrhythmia": [
            "Limit caffeine, alcohol, and other stimulants.",
            "Ensure adequate and consistent sleep patterns.",
            "Practice relaxation techniques to manage heart rate spikes."
        ]
    }

    selected_tips = set(tips["General"])

    if glucose > 125:
        results.append("⚠️ **High Blood Sugar (Hyperglycemia)**: Likely indicative of Diabetes, which significantly increases heart disease risk.")
        for tip in tips["Diabetes (High Glucose)"]: selected_tips.add(tip)
    elif glucose > 100:
        results.append("🟡 **Pre-diabetic Sugar Levels**: Monitoring is advised.")
        for tip in tips["Diabetes (High Glucose)"]: selected_tips.add(tip)

    if bpm > 100:
        risks.append("Arrhythmia (Tachycardia)")
        risks.append("Heart Failure Risk")
        for tip in tips["Arrhythmia"]: selected_tips.add(tip)
        for tip in tips["Heart Failure"]: selected_tips.add(tip)
    elif bpm < 60:
        risks.append("Arrhythmia (Bradycardia)")
        for tip in tips["Arrhythmia"]: selected_tips.add(tip)

    # Activity Context Logic
    if activity_context in ["Jogging", "Running", "Cycling", "Climbing Stairs"]:
        if bpm > 100:
            results.append(f"🏃 **Activity Context**: High BPM of {bpm} is expected during {activity_context}.")
    elif activity_context in ["Sitting/Relaxing", "Lying Down", "Standing Still"]:
        if bpm > 100:
            results.append(f"⚠️ **At Rest Warning**: A heart rate of {bpm} BPM while {activity_context} may require clinical review.")

    if "Chest Pain" in symptoms:
        risks.append("Coronary Artery Disease (Angina)")
        for tip in tips["Coronary Artery Disease (CAD)"]: selected_tips.add(tip)
    if "Shortness of Breath" in symptoms:
        risks.append("Heart Failure")
        for tip in tips["Heart Failure"]: selected_tips.add(tip)
    if "Swelling in Legs/Feet" in symptoms:
        risks.append("Congestive Heart Failure (Edema)")
        for tip in tips["Heart Failure"]: selected_tips.add(tip)
    if "Palpitations" in symptoms:
        risks.append("Arrhythmia / Atrial Fibrillation")
        for tip in tips["Arrhythmia"]: selected_tips.add(tip)
    if "Dizziness/Fainting" in symptoms:
        risks.append("Valve Disease or Low Cardiac Output")


    unique_risks = list(set(risks))

    # Rationale logic: Map risks to the statistics that triggered them
    rationale = []
    if glucose > 125: rationale.append(f"Glucose ({glucose}) > 125 mg/dL indicates Diabetes/Hyperglycemia risk.")
    elif glucose > 100: rationale.append(f"Glucose ({glucose}) > 100 mg/dL indicates Pre-diabetic levels.")

    if bpm > 100: rationale.append(f"Heart Rate ({bpm}) > 100 BPM indicates Tachycardia.")
    elif bpm < 60: rationale.append(f"Heart Rate ({bpm}) < 60 BPM indicates Bradycardia.")

    for symptom in symptoms:
        if symptom == "Chest Pain": rationale.append("Chest Pain is a primary indicator of Coronary Artery Disease.")
        if symptom == "Shortness of Breath": rationale.append("Shortness of Breath is a primary indicator of Heart Failure.")
        if symptom == "Swelling in Legs/Feet": rationale.append("Swelling (Edema) is often linked to Congestive Heart Failure.")
        if symptom == "Palpitations": rationale.append("Palpitations are linked to Arrhythmia.")


    if not unique_risks and risk_score < 30:
        message = "✅ No immediate heart disease indicators found based on current symptoms and profile."
        status = "Normal Health Indicators"
    else:
        message = "🚩 Potential Indicators detected. High correlation with recorded medical patterns."
        status = "Consultation Recommended"

    return status, message, results, list(selected_tips), risk_score, unique_risks, rationale


# Header
st.title("❤️ CardioGuard AI")
st.markdown("<h2 style='color: #000000 !important; font-weight: 950; margin-top: -10px;'>Advanced Heart Disease Analysis & Prediction</h2>", unsafe_allow_html=True)
st.write("Enter patient vitals and symptoms below for a preliminary heart health analysis.")


# Sidebar for Patient Demographics
with st.sidebar:
    st.header("Patient Profile")
    age = st.number_input("Age", min_value=1, max_value=120, value=30)
    gender = st.selectbox("Gender", ["Male", "Female", "Other"])
    weight = st.number_input("Weight (kg)", min_value=10, max_value=250, value=70)
    st.divider()
    st.header("🗂️ Advanced Health History")
    col_hist1, col_hist2 = st.columns(2)
    with col_hist1:
        smoker = st.toggle("Chain Smoker", False)
        tobacco = st.toggle("Other Tobacco Prods", False)
        obese = st.toggle("Clinically Obese", False)
        metabolic = st.toggle("Metabolic Syndrome", False)
        resp = st.toggle("Respiratory Illness", False)
    with col_hist2:
        stimulants = st.toggle("Stimulant Drug Use", False)
        fam_history = st.toggle("Family History", True)
        preeclampsia = st.toggle("History of Preeclampsia", False)
        cabg = st.toggle("CABG History", False)
        diabetes_history = st.toggle("Diagnosed Diabetes", False)


    st.divider()
    st.header("🏃 Live Activity Context")
    st.write("Simulate motion sensor data (Chest Accelerometer)")
    accel_x = st.slider("Chest Accel X", -20.0, 20.0, 2.0)
    accel_y = st.slider("Chest Accel Y", -20.0, 20.0, -9.0)
    accel_z = st.slider("Chest Accel Z", -20.0, 20.0, 0.5)

    current_activity = get_activity_context(accel_x, accel_y, accel_z)
    st.success(f"Predicted Activity: **{current_activity}**")

    st.divider()
    st.info("Note: This tool is for educational purposes and not a substitute for professional medical advice.")


# Main Input Section
col1, col2 = st.columns(2)


with col1:
    st.markdown("#### 🩺 Clinical Vitals")
    bpm = st.slider("Heart Rate (BPM)", 40, 200, 72)
    glucose = st.number_input("Blood Glucose Level (mg/dL)", min_value=50, max_value=500, value=90)
    blood_pressure = st.text_input("Blood Pressure (e.g., 120/80)", "120/80")


with col2:
    st.markdown("#### 🚩 Symptoms")
    symptoms_list = [
        "Chest Pain",
        "Shortness of Breath",
        "Swelling in Legs/Feet",
        "Palpitations",
        "Dizziness/Fainting",
        "Fatigue",
        "Nausea",
        "Sweating"
    ]
    selected_symptoms = st.multiselect("Select all that apply:", symptoms_list)


# Prediction Button
if st.button("Analyze Heart Health"):
    # Prepare risk model data
    # [Gender, Chain_smoker, Consumes_other_tobacco_products, HighBP, Obese, Diabetes, Metabolic_syndrome, Use_of_stimulant_drugs, Family_history, History_of_preeclampsia, CABG_history, Respiratory_illness]
    gender_map = {"Male": 1, "Female": 2, "Other": 0}
    high_bp = 1 if int(blood_pressure.split('/')[0]) > 140 else 0
    risk_inputs = [
        gender_map[gender], int(smoker), int(tobacco), high_bp, int(obese),
        1 if glucose > 125 or diabetes_history else 0,
        int(metabolic), int(stimulants), int(fam_history),
        int(preeclampsia), int(cabg), int(resp)
    ]

    status, main_msg, sugar_notes, health_tips, risk_score, concerns, rationale = predict_heart_disease(bpm, glucose, selected_symptoms, age, current_activity, risk_inputs)

    st.divider()

    # Results Presentation
    st.subheader("📊 Analysis Results")

    res_col1, res_col2 = st.columns([2, 1])

    with res_col1:
        st.markdown(f"""
        <div class="prediction-card">
            <h2 style='color: {"#ff4b4b" if "Normal" not in status else "#00ff88"}'>{status}</h2>
            <p style='font-size: 1.1em;'>{main_msg}</p>
            <hr>
            <div style="display: flex; justify-content: space-between; align-items: center;">
                <span style="font-size: 1.2em; font-weight: bold;">Data-Driven Risk Score:</span>
                <span style="font-size: 2em; color: {'#ff4b4b' if risk_score > 30 else '#00ff88'};">{risk_score}%</span>
            </div>
            <p style="font-size: 0.8em; opacity: 0.7; margin-top: 5px;">*Score based on patterns in 880+ clinical records.</p>
        </div>
        """, unsafe_allow_html=True)


        if concerns:
            st.markdown("#### 🚩 Specific Concerns Identified")
            concerns_html = "".join([f'<div style="background: rgba(255, 75, 75, 0.1); padding: 10px; border-radius: 10px; margin-bottom: 5px; border-left: 5px solid #ff4b4b;">{c}</div>' for c in concerns])
            st.markdown(concerns_html, unsafe_allow_html=True)

        if rationale:
            st.markdown("#### 🧐 Why this result? (Statistical Rationale)")
            for r in rationale:
                st.info(r)

        if sugar_notes:
            st.markdown("#### 🍭 Sugar Level Observations")
            for note in sugar_notes:
                st.write(note)


        # Health Tips Section
        st.markdown("#### 🥗 Recovery & Health Tips")
        tips_html = "<ul>" + "".join([f"<li>{tip}</li>" for tip in health_tips]) + "</ul>"
        st.markdown(f"""
        <div class="prediction-card" style="background: rgba(0, 255, 136, 0.05);">
            {tips_html}
        </div>
        """, unsafe_allow_html=True)


        # Download Precautions Button
        tips_text = "RECOMMENDED PRECAUTIONS & HEALTH TIPS\n" + "="*40 + "\n\n" + "\n".join([f"- {tip}" for tip in health_tips])
        st.download_button(
            label="📄 Download Precautions as TXT",
            data=tips_text,
            file_name="heart_health_precautions.txt",
            mime="text/plain",
            use_container_width=True
        )

    with res_col2:
        st.markdown("#### 📈 Vitals Summary")
        st.write(f"**BPM:** {bpm}")
        st.write(f"**Glucose:** {glucose} mg/dL")
        st.write(f"**Activity:** {current_activity}")
        st.write(f"**Symptoms Count:** {len(selected_symptoms)}")

    st.warning("⚠️ **Disclaimer:** This analysis is based on medical heuristics and does not constitute a clinical diagnosis. Please consult a qualified healthcare professional.")


# Footer
st.markdown("<br><br>", unsafe_allow_html=True)
st.markdown("<p style='text-align: center; color: rgba(255,255,255,0.6);'>Built with ❤️ by CardioGuard AI Team</p>", unsafe_allow_html=True)


Writing app.py


In [4]:
# Create dummy models
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# 1. Create activity prediction model (activity_model.joblib)
print("Creating activity prediction model...")
# Train a simple classifier on dummy accelerometer data
X_activity = np.random.randn(1000, 3)  # 1000 samples, 3 features (x, y, z accelerometer)
y_activity = np.random.randint(0, 13, 1000)  # 13 activity classes (0-12)

activity_model = RandomForestClassifier(n_estimators=50, random_state=42)
activity_model.fit(X_activity, y_activity)
joblib.dump(activity_model, 'activity_model.joblib')
print("✓ Created activity_model.joblib")

# 2. Create risk prediction model (risk_model.joblib)
print("\nCreating risk prediction model...")
# Train a binary classifier for heart disease risk
# Features: [Gender, Chain_smoker, Tobacco, HighBP, Obese, Diabetes, Metabolic, Stimulants, Family_history, Preeclampsia, CABG, Respiratory]
X_risk = np.random.randint(0, 3, (880, 12))  # 880 samples, 12 features
y_risk = np.random.randint(0, 2, 880)  # Binary: 0=No Risk, 1=Under Risk

risk_model = LogisticRegression(random_state=42)
risk_model.fit(X_risk, y_risk)
joblib.dump(risk_model, 'risk_model.joblib')
print("✓ Created risk_model.joblib")

print("\n✅ Model files created successfully!")

# Upload your background image
print("\n📤 Now upload your heart_bg.jpg:")
from google.colab import files
uploaded = files.upload()

print("\n✅ All required files are ready!")


Creating activity prediction model...
✓ Created activity_model.joblib

Creating risk prediction model...
✓ Created risk_model.joblib

✅ Model files created successfully!

📤 Now upload your heart_bg.jpg:


Saving WhatsApp Image 2026-01-05 at 12.02.33 AM.jpeg to WhatsApp Image 2026-01-05 at 12.02.33 AM.jpeg

✅ All required files are ready!


In [8]:
# Kill any existing streamlit processes
!pkill -9 streamlit

# Start Streamlit and wait
import subprocess
import time

print("Starting Streamlit server...")
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])

# Wait longer for server to fully start
time.sleep(10)

# Verify it's running
!curl http://localhost:8501
print("\n✅ Streamlit server is running on port 8501")


Starting Streamlit server...
<!--
 Copyright (c) Streamlit Inc. (2018-2022) Snowflake Inc. (2022-2025)

 Licensed under the Apache License, Version 2.0 (the "License");
 you may not use this file except in compliance with the License.
 You may obtain a copy of the License at

     http://www.apache.org/licenses/LICENSE-2.0

 Unless required by applicable law or agreed to in writing, software
 distributed under the License is distributed on an "AS IS" BASIS,
 WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
 See the License for the specific language governing permissions and
 limitations under the License.
-->

<!DOCTYPE html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <meta
      name="viewport"
      content="width=device-width, initial-scale=1, shrink-to-fit=no"
    />
    <link rel="shortcut icon" href="./favicon.png" />
    <link
      rel="preload"
      href="./static/media/SourceSansVF-Upright.ttf.BsWL4Kly.woff2"
      as="font"
      type

In [ ]:
!pip install pyngrok -q

from pyngrok import ngrok

# Create tunnel
public_url = ngrok.connect(8501)

print("\n" + "="*70)
print(f"🌐 Your app URL: {public_url}")
print("="*70)
print("\n✅ Click the URL above to access your app!")